In [1]:
import pandas as pd
import numpy as np
import glob
import joblib
import os
import warnings
warnings.filterwarnings('ignore')

from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout, Input
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.optimizers import Adam

# ── Load Dogecoin ─────────────────────────────────────────────────────────
files    = glob.glob("./data/*.csv")
doge_file = [f for f in files if 'Dogecoin' in f][0]

doge = pd.read_csv(doge_file)
doge["Date"] = pd.to_datetime(doge["Date"])
doge = doge.sort_values("Date").reset_index(drop=True)
print(f"Dogecoin rows: {len(doge)}")

Dogecoin rows: 2760


In [2]:
# ── Feature Engineering ────────────────────────────────────────────────────────
doge["lag_1"]         = doge["Close"].shift(1)
doge["lag_2"]         = doge["Close"].shift(2)
doge["lag_3"]         = doge["Close"].shift(3)
doge["SMA_7"]         = doge["Close"].rolling(7).mean()
doge["SMA_14"]        = doge["Close"].rolling(14).mean()
doge["EMA_12"]        = doge["Close"].ewm(span=12, adjust=False).mean()
doge["EMA_26"]        = doge["Close"].ewm(span=26, adjust=False).mean()
doge["MACD"]          = doge["EMA_12"] - doge["EMA_26"]
delta                = doge["Close"].diff()
gain                 = delta.clip(lower=0).rolling(14).mean()
loss                 = (-delta.clip(upper=0)).rolling(14).mean()
doge["RSI"]           = 100 - (100 / (1 + gain / (loss + 1e-10)))
doge["log_return"]    = np.log(doge["Close"] / doge["Close"].shift(1))
doge["rolling_std_7"] = doge["Close"].pct_change().rolling(7).std()
doge["momentum_7"]    = doge["Close"] - doge["Close"].shift(7)


# Target = next day log return
doge["target"] = doge["log_return"].shift(-1)
doge = doge.dropna().reset_index(drop=True)
print(f"Rows after engineering: {len(doge)}")

Rows after engineering: 2745


In [3]:
# ── Split & Scale ──────────────────────────────────────────────────────────────
features = [
    "Open", "High", "Low", "Volume",
    "lag_1", "lag_2", "lag_3",
    "SMA_7", "SMA_14", "EMA_12", "EMA_26",
    "MACD", "RSI", "log_return",
    "rolling_std_7", "momentum_7"
]

split = int(len(doge) * 0.8)

X_train = doge[features].iloc[:split]
X_test  = doge[features].iloc[split:]
y_train = doge["target"].iloc[:split].values
y_test  = doge["target"].iloc[split:].values

scaler         = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

os.makedirs("../models", exist_ok=True)
print(f"Train: {X_train_scaled.shape}, Test: {X_test_scaled.shape}")

Train: (2196, 16), Test: (549, 16)


In [4]:
# ── Linear Regression ──────────────────────────────────────────────────────────
model = LinearRegression()
model.fit(X_train_scaled, y_train)

lr_pred = model.predict(X_test_scaled)
lr_rmse = np.sqrt(mean_squared_error(y_test, lr_pred))
lr_mae  = mean_absolute_error(y_test, lr_pred)
lr_mda  = np.mean(np.sign(y_test) == np.sign(lr_pred)) * 100

print(f"=== Linear Regression — Dogecoin ===")
print(f"RMSE : {lr_rmse:.6f}")
print(f"MAE  : {lr_mae:.6f}")
print(f"MDA  : {lr_mda:.2f}%")



=== Linear Regression — Dogecoin ===
RMSE : 0.828388
MAE  : 0.267816
MDA  : 50.82%


In [5]:
# ── Random Forest ──────────────────────────────────────────────────────────────
from sklearn.ensemble import RandomForestRegressor

# RF ke liye alag scaler
rf_scaler      = StandardScaler()
X_train_rf_sc  = rf_scaler.fit_transform(X_train)
X_test_rf_sc   = rf_scaler.transform(X_test)

rf_model = RandomForestRegressor(
    n_estimators=500,
    max_depth=6,
    min_samples_leaf=10,
    max_features=0.6,
    random_state=42,
    n_jobs=-1
)
rf_model.fit(X_train_rf_sc, y_train)

rf_pred = rf_model.predict(X_test_rf_sc)
rf_rmse = np.sqrt(mean_squared_error(y_test, rf_pred))
rf_mae  = mean_absolute_error(y_test, rf_pred)
rf_mda  = np.mean(np.sign(y_test) == np.sign(rf_pred)) * 100

print(f"=== Random Forest — Dogecoin ===")
print(f"RMSE : {rf_rmse:.6f}  (LR: {lr_rmse:.6f})")
print(f"MAE  : {rf_mae:.6f}  (LR: {lr_mae:.6f})")
print(f"MDA  : {rf_mda:.2f}%  (LR: {lr_mda:.2f}%)")
print(f"       {'✓ Target MET' if rf_mda >= 55 else '— below 55%'}")



=== Random Forest — Dogecoin ===
RMSE : 0.114422  (LR: 0.828388)
MAE  : 0.054486  (LR: 0.267816)
MDA  : 52.46%  (LR: 50.82%)
       — below 55%


In [6]:
# ── LSTM Feature Engineering ───────────────────────────────────────────────────
def compute_rsi(series, period=14):
    delta = series.diff()
    gain  = delta.clip(lower=0).ewm(com=period-1, min_periods=period).mean()
    loss  = (-delta.clip(upper=0)).ewm(com=period-1, min_periods=period).mean()
    return 100 - (100 / (1 + gain / loss))

d = doge[['Date','Open','High','Low','Close','Volume']].copy().sort_values('Date').reset_index(drop=True)

d['EMA_12']      = d['Close'].ewm(span=12, adjust=False).mean()
d['EMA_26']      = d['Close'].ewm(span=26, adjust=False).mean()
d['MACD']        = d['EMA_12'] - d['EMA_26']
d['RSI']         = compute_rsi(d['Close'])
bb_mid           = d['Close'].rolling(20).mean()
bb_std           = d['Close'].rolling(20).std()
d['BB_Position'] = (d['Close'] - (bb_mid - 2*bb_std)) / (4*bb_std + 1e-9)
tr               = pd.concat([
                       d['High'] - d['Low'],
                       (d['High'] - d['Close'].shift(1)).abs(),
                       (d['Low']  - d['Close'].shift(1)).abs()
                   ], axis=1).max(axis=1)
d['ATR_14']      = tr.rolling(14).mean()
d['Log_Return']  = np.log(d['Close'] / d['Close'].shift(1))
d['target']      = d['Log_Return'].shift(-1)
d = d.replace([np.inf, -np.inf], np.nan).dropna().reset_index(drop=True)

LSTM_FEATURES = ['Close', 'Volume', 'RSI', 'MACD', 'BB_Position', 'ATR_14', 'Log_Return']
print(f"LSTM rows: {len(d)}, Features: {len(LSTM_FEATURES)}")

LSTM rows: 2725, Features: 7


In [7]:
# ── Split, Scale, Sequences ────────────────────────────────────────────────────
WINDOW = 75

f_scaler = MinMaxScaler()
X_all    = f_scaler.fit_transform(d[LSTM_FEATURES])
targets  = d['target'].values

def make_sequences(X, y):
    X_seq, y_seq = [], []
    for i in range(WINDOW, len(X)):
        X_seq.append(X[i-WINDOW:i])
        y_seq.append(y[i])
    return np.array(X_seq), np.array(y_seq)

X_seq, y_seq = make_sequences(X_all, targets)
split_seq    = int(len(X_seq) * 0.8)

X_train_s, y_train_s = X_seq[:split_seq], y_seq[:split_seq]
X_test_s,  y_test_s  = X_seq[split_seq:], y_seq[split_seq:]
print(f"Train: {X_train_s.shape}, Test: {X_test_s.shape}")

Train: (2120, 75, 7), Test: (530, 75, 7)


In [8]:
# ── Build & Train ──────────────────────────────────────────────────────────────
import math
from tensorflow.keras.callbacks import LearningRateScheduler

def cosine_schedule(epoch, lr):
    return 1e-6 + 0.5 * (3e-4 - 1e-6) * (1 + math.cos(math.pi * epoch / 100))

lstm_model = Sequential([
    Input(shape=(WINDOW, len(LSTM_FEATURES))),
    LSTM(64, return_sequences=True),  Dropout(0.3),
    LSTM(32, return_sequences=False), Dropout(0.2),
    Dense(16, activation='relu'),
    Dense(1)
])
lstm_model.compile(optimizer=Adam(3e-4), loss='mse')
lstm_model.fit(
    X_train_s, y_train_s,
    epochs=100, batch_size=16,
    validation_data=(X_test_s, y_test_s),
    callbacks=[
        EarlyStopping(monitor='val_loss', patience=20,
                      restore_best_weights=True, verbose=1),
        LearningRateScheduler(cosine_schedule, verbose=0)
    ],
    verbose=1
)
lstm_model.save("models/doge_lstm.keras")

Epoch 1/100



  1/133 ━━━━━━━━━━━━━━━━━━━━ 1:40 760ms/step - loss: 0.0058


  6/133 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step - loss: 0.0045   


 11/133 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step - loss: 0.0042


 16/133 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step - loss: 0.0041

...


127/133 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - loss: 0.0048


131/133 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - loss: 0.0048


133/133 ━━━━━━━━━━━━━━━━━━━━ 2s 13ms/step - loss: 0.0038 - val_loss: 0.0131 - learning_rate: 2.1415e-04


Epoch 37: early stopping


Restoring model weights from the end of the best epoch: 17.


In [9]:
# ── Evaluation ─────────────────────────────────────────────────────────────────
preds_log  = lstm_model.predict(X_test_s, verbose=0).flatten()
actual_log = y_test_s

lstm_rmse = np.sqrt(mean_squared_error(actual_log, preds_log))
lstm_mae  = mean_absolute_error(actual_log, preds_log)
lstm_mda  = np.mean(np.sign(actual_log) == np.sign(preds_log)) * 100

print(f"=== LSTM — Dogecoin (Log Return) ===")
print(f"RMSE : {lstm_rmse:.6f}  (RF: {rf_rmse:.6f}  |  LR: {lr_rmse:.6f})")
print(f"MAE  : {lstm_mae:.6f}  (RF: {rf_mae:.6f}  |  LR: {lr_mae:.6f})")
print(f"MDA  : {lstm_mda:.2f}%  (RF: {rf_mda:.2f}%  |  LR: {lr_mda:.2f}%)")

=== LSTM — Dogecoin (Log Return) ===
RMSE : 0.113795  (RF: 0.114422  |  LR: 0.828388)
MAE  : 0.052705  (RF: 0.054486  |  LR: 0.267816)
MDA  : 51.13%  (RF: 52.46%  |  LR: 50.82%)
